# Initialisation

In [ ]:
import os
import numpy as np
import pandas as pd
import re

import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
os.chdir('drive/MyDrive/Colab Notebooks/IMDB_movie_set')
!pwd

/content/drive/MyDrive/Colab Notebooks/IMDB_movie_set


# Data Preprocessing

In [ ]:
df = pd.read_csv('IMDB Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
df.duplicated().sum()

418

In [ ]:
df.drop_duplicates(inplace=True)

Removing HTML Tags

In [ ]:
def remove_tags(raw_data):
  cleaned_text = re.sub(re.compile('<.*?>'), '', raw_data)
  return cleaned_text

In [ ]:
df['review'] = df['review'].apply(remove_tags)
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. The filming tec...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


Lowercase

In [ ]:
df['review'] = df['review'].apply(lambda x: x.lower())
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. the filming tec...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


In [ ]:
def remove_stopwords(data):
  new_data = []

  for word in data.split():
    if word in stopwords.words('english'):
      new_data.append('')
    else:
      new_data.append(word)
  x = new_data[:]
  new_data.clear()
  return " ".join(x)

In [ ]:
df['review'] = df['review'].apply(remove_stopwords)
df.head()

,review,sentiment
0,one reviewers mentioned watching 1 oz e...,positive
1,wonderful little production. filming techniq...,positive
2,thought wonderful way spend time hot s...,positive
3,basically there's family little boy (jake) ...,negative
4,"petter mattei's ""love time money"" visuall...",positive


# Word2Vec

In [ ]:
!pip install gensim

from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize

In [ ]:
nltk.download('punkt_tab')

df['review_tokenized'] = df['review'].apply(lambda x: word_tokenize(x))
df.head()

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


,review,sentiment,review_tokenized
0,one reviewers mentioned watching 1 oz e...,positive,"[one, reviewers, mentioned, watching, 1, oz, e..."
1,wonderful little production. filming techniq...,positive,"[wonderful, little, production, ., filming, te..."
2,thought wonderful way spend time hot s...,positive,"[thought, wonderful, way, spend, time, hot, su..."
3,basically there's family little boy (jake) ...,negative,"[basically, there, 's, family, little, boy, (,..."
4,"petter mattei's ""love time money"" visuall...",positive,"[petter, mattei, 's, ``, love, time, money, ''..."


In [ ]:
model = Word2Vec(sentences=df['review_tokenized'], vector_size=100, window=5, min_count=5, workers=4)

In [ ]:
def get_review_vector(tokens, model):
  vector = np.zeros(100)
  count = 0

  for word in tokens:
    if word in model.wv:
      vector += model.wv[word]
      count += 1

  if count > 0:
    vector /= count

  return vector

In [ ]:
df['review_vector'] = df['review_tokenized'].apply(lambda x: get_review_vector(x, model))
df.head()

,review,sentiment,review_tokenized,review_vector
0,one reviewers mentioned watching 1 oz e...,positive,"[one, reviewers, mentioned, watching, 1, oz, e...","[-0.3851907453930085, -0.39358612385432795, -0..."
1,wonderful little production. filming techniq...,positive,"[wonderful, little, production, ., filming, te...","[-0.4032396433594664, -0.37893754146564795, -0..."
2,thought wonderful way spend time hot s...,positive,"[thought, wonderful, way, spend, time, hot, su...","[-0.34853497488551183, -0.4425977544562722, -0..."
3,basically there's family little boy (jake) ...,negative,"[basically, there, 's, family, little, boy, (,...","[-0.5542528208524707, -0.4333133723822434, -0...."
4,"petter mattei's ""love time money"" visuall...",positive,"[petter, mattei, 's, ``, love, time, money, ''...","[-0.3323621218558401, -0.5172836823677965, -0...."


In [ ]:
X = np.array(df['review_vector'].tolist())
y = df['sentiment'].apply(lambda x: 1 if x == 'positive' else 0).values

Dividing Dataset into Training and Testing Modules

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model: Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

rf_model.fit(X_train.tolist(), y_train)

y_pred = rf_model.predict(X_test.tolist())

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

report = classification_report(y_test, y_pred)
print("Classification Report:\n", report)

Accuracy: 0.8238378541897752
Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.81      0.82      4939
           1       0.82      0.84      0.83      4978

    accuracy                           0.82      9917
   macro avg       0.82      0.82      0.82      9917
weighted avg       0.82      0.82      0.82      9917



In [42]:
!git clone https://github.com/RiteshRajMoond/Text_Classification_using_ML.git

Cloning into 'Text_Classification_using_ML'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.


In [43]:
!pip freeze > requirements.txt

# Testing Manually

In [45]:
def preprocess_review(review):
  review = remove_tags(review)
  review = review.lower()
  review = remove_stopwords(review)
  tokens = word_tokenize(review)
  vector = get_review_vector(tokens, model)
  return vector

def predict_sentiment(review, model, threshold=0.5):
  review_vector = preprocess_review(review)
  review_vector = np.array(review_vector).reshape(1, -1)
  prediction = model.predict(review_vector)
  sentiment = 'positive' if prediction[0] == 1 else 'negative'
  return sentiment

In [46]:
review = "I Loved the movie"
result = predict_sentiment(review, rf_model)
print(result)

positive
